# LHC alignment: PCC vs PCC-tiled (translation only)

This notebook compares the original global phase-correlation alignment (`pcc`) against a more robust tiled phase-correlation (`pcc_tiled`) **without allowing rotation**.

It produces:
- per-pair shift tables and quality stats
- centroid NN-distance diagnostics (before/after)
- visual overlays for problematic transitions (e.g. OM3→OM4, OM4→OM5)

In [ ]:
import importlib
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'src').exists() and (p / 'input').exists() and (p / 'output').exists():
            return p
    raise RuntimeError(f'Could not find repo root from start={start}')

# Make sure we import the workspace version of tree_tracking.py
ROOT = find_repo_root(Path.cwd())
APP_DIR = ROOT / 'src' / 'flask_app_tracking'
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

import tree_tracking
importlib.reload(tree_tracking)
from tree_tracking import TreeTrackingGraph

print('ROOT:', ROOT)
print('Imported TreeTrackingGraph from:', APP_DIR)

In [ ]:
# LHC inputs (chronological; excluding 9/12/25 due to gross misalignment)
MULTITHRESH_DIR = ROOT / 'output' / 'input_crowns_multithreshold_lhc_17Mar26'
ORTHO_DIR = ROOT / 'input' / 'input_om_lhc'

VALID_LHC_STEMS = [
    'odm_orthophoto25_10_25',
    'odm_orthophoto9_11_25',
    'odm_orthophoto20_11_25',
    'odm_orthophoto26_11_25',
    'odm_orthophoto11_1_26',
    'odm_orthophoto4_02_26',
    'odm_orthophoto20_02_26',
    'odm_orthophoto7_03_26',
 ]

pairs = []
missing = []
for stem in VALID_LHC_STEMS:
    gpkg = MULTITHRESH_DIR / f'{stem}_multithreshold.gpkg'
    tif = ORTHO_DIR / f'{stem}.tif'
    if gpkg.exists() and tif.exists():
        pairs.append((str(gpkg), str(tif), stem))
    else:
        missing.append({'stem': stem, 'gpkg_exists': gpkg.exists(), 'tif_exists': tif.exists()})

if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

print('Found pairs:', len(pairs))
print('MULTITHRESH_DIR:', MULTITHRESH_DIR)
print('ORTHO_DIR:', ORTHO_DIR)
for i, (_, _, stem) in enumerate(pairs, start=1):
    print(f'  OM{i}: {stem}')

In [ ]:
# Alignment runner + diagnostics helpers
from shapely.affinity import translate
from sklearn.neighbors import NearestNeighbors

THRESH_TAG = 'conf_0p65'  # used for centroid-based diagnostics only
BASE_THRESHOLD_TAG = 'conf_0p45'  # what the pipeline loads into tracker.crowns_gdfs
MAX_PREVIEW = 1200

def make_tracker(output_subdir: str, resize_factor: float = 0.10) -> TreeTrackingGraph:
    out_dir = ROOT / 'output' / output_subdir
    out_dir.mkdir(parents=True, exist_ok=True)
    tr = TreeTrackingGraph(
        auto_discover=False,
        multithresh_dir=str(MULTITHRESH_DIR),
        ortho_dir=str(ORTHO_DIR),
        output_dir=str(out_dir),
        simplify_tol=1.0,
        resize_factor=resize_factor,
        max_crowns_preview=200,
    )
    tr.file_pairs = [(g, o) for g, o, _ in pairs]
    tr.om_ids = list(range(1, len(pairs) + 1))
    tr.base_threshold_tag = None
    return tr

def load_and_align(method: str) -> TreeTrackingGraph:
    tr = make_tracker(output_subdir=f'lhc_alignment_{method}_25Mar26')
    tr.load_multithreshold_data(
        base_threshold_tag=BASE_THRESHOLD_TAG,
        load_images=False,
        align=True,
        align_method=method,
        align_threshold_tag=THRESH_TAG,
    )
    return tr

def load_raw_threshold_gdfs(threshold_tag: str) -> dict:
    raw = {}
    for om_id, (gpkg, _, _) in enumerate(pairs, start=1):
        # layer read (GeoPackage layers are the threshold tags)
        try:
            gdf = gpd.read_file(gpkg, layer=threshold_tag)
        except Exception:
            gdf = gpd.GeoDataFrame(geometry=[], crs=None)
        raw[om_id] = gdf
    return raw

def centroids_xy(gdf: gpd.GeoDataFrame) -> np.ndarray:
    if gdf is None or gdf.empty:
        return np.zeros((0, 2), dtype=float)
    geoms = [g for g in gdf.geometry if g is not None and not g.is_empty]
    if not geoms:
        return np.zeros((0, 2), dtype=float)
    return np.array([[g.centroid.x, g.centroid.y] for g in geoms], dtype=float)

def median_nn(prev_xy: np.ndarray, curr_xy: np.ndarray) -> tuple[float, float]:
    if prev_xy.size == 0 or curr_xy.size == 0:
        return float('nan'), float('nan')
    nn = NearestNeighbors(n_neighbors=1).fit(prev_xy)
    d, _ = nn.kneighbors(curr_xy)
    d = d.reshape(-1)
    return float(np.median(d)), float(np.percentile(d, 90))

print('Helpers defined.')

In [ ]:
# Run both alignments (PCC baseline vs robust tiled PCC)
tracker_pcc = load_and_align('pcc')
tracker_tiled = load_and_align('pcc_tiled')

print('Loaded + aligned.')
print('Total crowns loaded (PCC):', sum(len(v) for v in tracker_pcc.crowns_gdfs.values()))
print('Total crowns loaded (PCC-tiled):', sum(len(v) for v in tracker_tiled.crowns_gdfs.values()))

print('\nCumulative shifts to OM1:')
for om_id in tracker_pcc.om_ids:
    dx1, dy1 = tracker_pcc.alignment_shifts.get(om_id, (0.0, 0.0))
    dx2, dy2 = tracker_tiled.alignment_shifts.get(om_id, (0.0, 0.0))
    print(f'  OM{om_id}: pcc(dx={dx1:+.2f}, dy={dy1:+.2f}) | tiled(dx={dx2:+.2f}, dy={dy2:+.2f})')

In [ ]:
# Quantitative diagnostics: centroid NN distances before/after, per consecutive OM pair
raw_gdfs = load_raw_threshold_gdfs(THRESH_TAG)

rows = []
for om_id in range(2, len(pairs) + 1):
    prev_id = om_id - 1
    prev_xy = centroids_xy(raw_gdfs[prev_id])
    curr_xy = centroids_xy(raw_gdfs[om_id])
    med_b, p90_b = median_nn(prev_xy, curr_xy)

    # Apply shifts for each method to the CURRENT OM only (to compare the step quality)
    dx_p, dy_p = tracker_pcc.alignment_shifts.get(om_id, (0.0, 0.0))
    dx_t, dy_t = tracker_tiled.alignment_shifts.get(om_id, (0.0, 0.0))
    curr_xy_p = curr_xy + np.array([dx_p, dy_p])[None, :] if curr_xy.size else curr_xy
    curr_xy_t = curr_xy + np.array([dx_t, dy_t])[None, :] if curr_xy.size else curr_xy

    med_ap, p90_ap = median_nn(prev_xy, curr_xy_p)
    med_at, p90_at = median_nn(prev_xy, curr_xy_t)

    dbg = tracker_tiled.alignment_debug.get(om_id, {})
    rows.append({
        'pair': f'OM{prev_id}->OM{om_id}',
        'med_nn_before': med_b,
        'med_nn_after_pcc': med_ap,
        'med_nn_after_tiled': med_at,
        'p90_before': p90_b,
        'p90_after_pcc': p90_ap,
        'p90_after_tiled': p90_at,
        'tiled_n_valid_tiles': dbg.get('n_valid_tiles', np.nan),
        'tiled_n_inliers': dbg.get('n_inliers', np.nan),
        'tiled_median_error_inliers': dbg.get('median_error_inliers', dbg.get('median_error_all', np.nan)),
        'tiled_fallback': dbg.get('fallback', ''),
    })

diag_df = pd.DataFrame(rows)
display(diag_df)

# Plot: median NN before vs after (two methods)
x = np.arange(len(diag_df))
w = 0.25
plt.figure(figsize=(14, 5))
plt.bar(x - w, diag_df['med_nn_before'], width=w, label='Before (raw)')
plt.bar(x, diag_df['med_nn_after_pcc'], width=w, label='After PCC')
plt.bar(x + w, diag_df['med_nn_after_tiled'], width=w, label='After PCC-tiled')
plt.xticks(x, diag_df['pair'], rotation=30, ha='right')
plt.ylabel('Distance (map units)')
plt.title(f'LHC alignment diagnostic using {THRESH_TAG} crown centroids')
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Plot: cumulative shifts (dx,dy) for both methods
oms = tracker_pcc.om_ids
dx_p = [tracker_pcc.alignment_shifts[o][0] for o in oms]
dy_p = [tracker_pcc.alignment_shifts[o][1] for o in oms]
dx_t = [tracker_tiled.alignment_shifts[o][0] for o in oms]
dy_t = [tracker_tiled.alignment_shifts[o][1] for o in oms]

plt.figure(figsize=(12, 4))
plt.plot(oms, dx_p, '-o', label='PCC dx')
plt.plot(oms, dy_p, '-o', label='PCC dy')
plt.plot(oms, dx_t, '--o', label='Tiled dx')
plt.plot(oms, dy_t, '--o', label='Tiled dy')
plt.xlabel('OM id (sequence order)')
plt.ylabel('Shift (map units)')
plt.title('Cumulative alignment shifts to OM1')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Visual sanity-check: overlay low-res orthos for key problematic transitions
from skimage.transform import AffineTransform, warp

def shift_image_by_map_units(img: np.ndarray, dx: float, dy: float, ref_transform) -> np.ndarray:
    # Convert map-units translation to pixel translation using ref_transform
    # ref_transform.a is pixel width; ref_transform.e is pixel height (likely negative).
    tx = float(dx / ref_transform.a) if ref_transform.a != 0 else 0.0
    ty = float(dy / ref_transform.e) if ref_transform.e != 0 else 0.0
    # skimage AffineTransform translation is (x, y) in pixels; warp expects inverse mapping internally.
    tform = AffineTransform(translation=(tx, ty))
    out = warp(img, tform.inverse, order=1, preserve_range=True, mode='constant', cval=0.0)
    return out.astype(np.float32)

def show_pair_overlay(prev_path: str, curr_path: str, dx: float, dy: float, title: str, max_preview: int = 1200):
    prev, prev_tr = TreeTrackingGraph._read_ortho_lowres(prev_path, max_preview)
    curr, curr_tr = TreeTrackingGraph._read_ortho_lowres(curr_path, max_preview)
    # Crop to overlap for visualization, then shift CURRENT to align with PREV
    prev_crop, curr_crop = TreeTrackingGraph._crop_to_overlap(prev, curr, prev_tr, curr_tr)
    if prev_crop is None or curr_crop is None:
        print(title, '-> no overlap at lowres')
        return
    prev_crop, curr_crop = TreeTrackingGraph._match_shape(prev_crop, curr_crop)
    curr_shifted = shift_image_by_map_units(curr_crop, dx=dx, dy=dy, ref_transform=prev_tr)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(prev_crop, cmap='gray')
    axes[0].set_title('Prev (ref)')
    axes[1].imshow(curr_crop, cmap='gray')
    axes[1].set_title('Curr (raw)')
    axes[2].imshow(prev_crop, cmap='gray')
    axes[2].imshow(curr_shifted, cmap='magma', alpha=0.45)
    axes[2].set_title('Overlay: ref + shifted curr')
    for ax in axes:
        ax.axis('off')
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

# Choose the pairs we care about most (historically problematic)
KEY_PAIRS = [(3, 4), (4, 5), (5, 6)]

for prev_id, curr_id in KEY_PAIRS:
    prev_path = pairs[prev_id - 1][1]
    curr_path = pairs[curr_id - 1][1]
    dxp, dyp = tracker_pcc.alignment_shifts.get(curr_id, (0.0, 0.0))
    dxt, dyt = tracker_tiled.alignment_shifts.get(curr_id, (0.0, 0.0))
    show_pair_overlay(prev_path, curr_path, dx=dxp, dy=dyp, title=f'OM{prev_id}->OM{curr_id} overlay (PCC)')
    show_pair_overlay(prev_path, curr_path, dx=dxt, dy=dyt, title=f'OM{prev_id}->OM{curr_id} overlay (PCC-tiled)')

In [ ]:
# Inspect PCC-tiled internals: how many tiles were accepted per step, and why rejected
dbg_rows = []
for om_id in range(2, len(pairs) + 1):
    d = tracker_tiled.alignment_debug.get(om_id, {})
    rej = d.get('rejected', {}) if isinstance(d.get('rejected', {}), dict) else {}
    dbg_rows.append({
        'om_id': om_id,
        'pair': f'OM{om_id-1}->OM{om_id}',
        'n_tile_positions': d.get('n_tile_positions', np.nan),
        'n_valid_tiles': d.get('n_valid_tiles', np.nan),
        'n_inliers': d.get('n_inliers', np.nan),
        'median_error_inliers': d.get('median_error_inliers', d.get('median_error_all', np.nan)),
        'fallback': d.get('fallback', ''),
        'rej_low_texture': rej.get('low_texture', np.nan),
        'rej_high_error': rej.get('high_error', np.nan),
        'rej_large_shift': rej.get('too_large_shift', np.nan),
    })
dbg_df = pd.DataFrame(dbg_rows)
display(dbg_df)

print('If you see frequent fallback=full_pcc_insufficient_tiles, we can relax thresholds (min_texture_std / max_error) or reduce min_valid_tiles.')

In [ ]:
# Compact debug summary (small enough to read in output)
fallback_counts = dbg_df['fallback'].fillna('').value_counts()
print('Fallback counts:')
print(fallback_counts.to_string())

small_cols = [
    'pair', 'n_tile_positions', 'n_valid_tiles', 'n_inliers', 'median_error_inliers', 'fallback',
    'rej_low_texture', 'rej_high_error', 'rej_large_shift'
 ]
print('\nPer-step summary (key columns):')
print(dbg_df[small_cols].to_string(index=False))